## Create a disease driver heatmap

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
from matplotlib import rcParams

# verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.verbosity = 3               
sc.logging.print_version_and_date()

# making sure plots & clusters are reproducible
np.random.seed(42)

# custom functions
import sc_toolbox as sct

Running Scanpy 1.8.2, on 2025-01-27 12:58.


In [3]:
## path variables
adata_dir = '/mnt/smb/Niklas/GLPG_data_temp/240811_GLPG_annotated.h5ad'
data_dir = '/home/niklas/projects/GLPG_drug_perturbations/01_data/'
out_dir = '/home/niklas/projects/GLPG_drug_perturbations/02_figures/fig3/'

In [4]:
## plotting variables
sc.settings.figdir = '/home/niklas/projects/GLPG_drug_perturbations/02_figures/fig3/'
sc.set_figure_params(vector_friendly = True)
plt.rcParams['figure.figsize'] = (6, 5)
plt.rcParams['pdf.fonttype'] = 42

In [5]:
## load anndata object
adata = sc.read(adata_dir)

In [6]:
## have a look at the adata object: 442k cells x 24500 genes
adata

AnnData object with n_obs × n_vars = 442439 × 24500
    obs: 'identifier', 'time_point', 'n_counts', 'preprocessing_cluster', 'size_factors', 'S_score', 'G2M_score', 'phase', 'treatment_time', 'treatment', 'sample_name', 'condition', 'percent_mito', 'cluster_03', 'doublet_scores', 'n_genes', 'louvain_1', 'louvain_2', 'cell_type', 'group', 'ChromiumBatch', 'LibraryPrepBatch', 'SeqBatch', 'ashcroft_score', 'body_weight_d0', 'body_weight_d1', 'body_weight_d2', 'body_weight_d3', 'body_weight_d4', 'body_weight_d5', 'body_weight_d6', 'body_weight_d7', 'body_weight_d8', 'body_weight_d9', 'body_weight_d10', 'body_weight_d11', 'body_weight_d12', 'body_weight_d13', 'body_weight_d14', 'body_weight_d15', 'body_weight_d16', 'body_weight_d17', 'body_weight_d18', 'body_weight_d19', 'body_weight_d20', 'body_weight_d21', 'body_weight_min', 'body_weight_max', 'delta_max_body_weight', 'delta_body_weight_d7_10', 'delta_body_weight_d14_d21'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispe

### Identify cell types with high sampling error

In [12]:
df = pd.crosstab(adata.obs['cell_type'],adata.obs['sample_name'])
df.head(15)

sample_name,d10BLM9,d10BLM10,d10BLM11,d10BLM12,d10BLM13,d10BLM14,d10BLM15,d10BLM16,d10BLM_Combo39,d10BLM_Combo41,...,d21veh65,d21veh66,d100BLM_Combo48,d100BLM_G120528,d100BLM_G169038,d100BLM_Nint58,d210BLMCombo124,d210BLMG1690110,d210BLM_G120596,d210BLM_Nint139
cell_type,,,,,,,,,,,,,,,,,,,,,
AT1,8,1,1,1,11,19,17,22,1,0,...,0,0,38,22,4,2,7,1,12,13
AT2,0,0,0,0,0,3,0,0,0,0,...,98,156,0,0,0,0,0,2,0,4
AT2 activated,1,1,6,1,1,23,15,4,0,4,...,2,5,16,30,7,25,4,15,7,50
Krt8+ ADI,30,10,14,29,55,19,24,69,7,22,...,0,0,74,81,11,28,9,2,5,9
Basal,0,0,0,0,0,0,0,1,0,0,...,0,0,2,0,0,0,0,0,0,0
Club/Ciliated,2,2,3,1,4,10,6,1,0,0,...,0,4,27,2,0,13,0,0,1,6
Ciliated,12,1,19,11,31,23,20,8,8,73,...,21,30,16,30,29,9,20,11,5,28
NEC,0,0,1,0,0,0,0,0,0,2,...,0,1,0,0,3,2,0,0,0,0
Aerocyte capillary EC,7,6,20,12,68,20,43,47,9,31,...,132,79,90,79,36,48,53,44,38,109


In [13]:
## possible thresholds
## cell type detected in at least X samples
## cell type detected in at least X samples per treatment 
## X cell of this cell type detected in per sample

### Calculate and annotate relative cell type frequency table

In [6]:
## frequency table
xlabel = 'sample_name'
cell_types_label = 'cell_type'

In [7]:
freq_tab = sct.calc.relative_frequencies(adata, group_by = cell_types_label, xlabel = xlabel, condition = None)
freq_tab.head(5)

,AT1,AT2,AT2 activated,Krt8+ ADI,Basal,Club/Ciliated,Ciliated,NEC,Aerocyte capillary EC,Transitional capillary EC,...,Cd8+ T cells,Cd4+/Cd8+ T cells,T reg cells,Th2 cells,Th17 cells,Themis+ T cells,Proliferating T cells,NK T cells,NK cells,sample_name
d100BLM_Combo48,0.010585,0.0,0.004457,0.020613,0.000557,0.007521,0.004457,0.000000,0.025070,0.004178,...,0.001671,0.047354,0.006407,0.002507,0.022006,0.000279,0.001114,0.020891,0.008357,d100BLM_Combo48
d100BLM_G120528,0.005167,0.0,0.007046,0.019023,0.000000,0.000470,0.007046,0.000000,0.018553,0.003288,...,0.019962,0.026069,0.012682,0.004932,0.009864,0.000000,0.005402,0.035463,0.012212,d100BLM_G120528
d100BLM_G169038,0.001171,0.0,0.002049,0.003220,0.000000,0.000000,0.008489,0.000878,0.010539,0.000293,...,0.037763,0.008489,0.033665,0.006440,0.030152,0.000293,0.004684,0.067037,0.024883,d100BLM_G169038
d100BLM_Nint58,0.000569,0.0,0.007110,0.007964,0.000000,0.003697,0.002560,0.000569,0.013652,0.002275,...,0.000284,0.071388,0.012799,0.004551,0.010808,0.000000,0.002844,0.022184,0.006826,d100BLM_Nint58
d10BLM10,0.000574,0.0,0.000574,0.005737,0.000000,0.001147,0.000574,0.000000,0.003442,0.000000,...,0.053930,0.028686,0.024670,0.006311,0.023523,0.000000,0.008032,0.075731,0.014917,d10BLM10


In [8]:
## sample annotation table
sample_label = 'sample_name'
treatment_label = 'treatment'
time_label = 'time_point'
outcome_1 = 'ashcroft_score'
outcome_2 = 'delta_max_body_weight'

## extract annotation from anndata
sample_meta = adata.obs[[sample_label, treatment_label, time_label, outcome_1, outcome_2]]
sample_meta.reset_index(drop=True, inplace=True)
sample_meta.drop_duplicates(inplace=True)
sample_meta = sample_meta.set_index('sample_name')
sample_meta.head(5)

/home/niklas/miniconda3/envs/niche_fibrosis_env/lib/python3.8/site-packages/pandas/util/_decorators.py:311: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return func(*args, **kwargs)


,treatment,time_point,ashcroft_score,delta_max_body_weight
sample_name,,,,
d100BLM_Combo48,Combo,d10,3.0000,-7.6
d100BLM_G120528,GLPG1205,d10,2.5000,-2.0
d100BLM_G169038,GLPG1690,d10,2.6875,-4.1
d100BLM_Nint58,Nintedanib,d10,3.5000,-7.2
d10BLM_Combo39,Combo,d10,2.2500,-2.5


In [9]:
## concat dataframes
freq_tab = pd.concat([sample_meta, freq_tab], axis = 1)

In [10]:
## drop NAs before calculating correlation coefficients
freq_tab = freq_tab.dropna()

In [11]:
## drop uncessary sample_name column
freq_tab.drop(labels = 'sample_name', axis = 1, inplace = True)

In [12]:
### split dataframes based on time
#freq_tab_d10 = freq_tab[freq_tab['time_point'] == 'd10']
#freq_tab_d21 = freq_tab[freq_tab['time_point'] == 'd21']

In [13]:
#freq_tab_d10.head(5)

In [14]:
#freq_tab_d21.head(5)

### Calculate correlation between cell type frequencies and disease severity

In [15]:
## 'delta_max_body_weight' is by design negative for most samples, so correlation coefficient will be negative
## while for ashcroft score it will be positive
## if we want cell types associated with high ashcroft score and high body weigh to cluster together
## we have to rename and multiply with -1
freq_tab['max_body_weight_loss'] = freq_tab['delta_max_body_weight']*(-1)
freq_tab.drop(labels = 'delta_max_body_weight', axis = 1, inplace = True)

In [16]:
freq_tab.columns

Index(['treatment', 'time_point', 'ashcroft_score', 'AT1', 'AT2',
       'AT2 activated', 'Krt8+ ADI', 'Basal', 'Club/Ciliated', 'Ciliated',
       'NEC', 'Aerocyte capillary EC', 'Transitional capillary EC',
       'General capillary EC', 'Proliferating capillary EC',
       'Vwa1+/Col15a1+ ectopic capillary EC', 'Arterial EC', 'Venous EC',
       'Lymphatic EC', 'Adventitial Fibroblasts', 'Adh7+ Fibroblasts',
       'Lgr5+/Lgr6+ Fibroblasts', 'Lgr5+/Lgr6+ Fibroblasts activated',
       'Lipofibroblasts', 'Lipo/Myofibroblast Transition',
       'Cthrc1+ Myofibroblasts', 'Spp1+ Myofibroblasts', 'Pericytes',
       'Pericytes activated', 'aSMCs', 'vSMCs', 'Mesothelial',
       'Non-classical Monocytes', 'Classical Monocytes',
       'Intermediate Monocytes', 'Transitional Monocytes', 'M2 Macrophages',
       'AM', 'Proliferating AM', 'Prg4+ IM', 'Lyve1+/Cd163+ IM',
       'Lyve1-/Cd163- IM', 'Proliferating IM', 'cDC1', 'cDC2', 'pDC',
       'Proliferating cDC1', 'Proliferating cDC2', 'M

In [17]:
## extract cell type columns
cell_types = freq_tab.columns[3:-1:1]
cell_types

Index(['AT1', 'AT2', 'AT2 activated', 'Krt8+ ADI', 'Basal', 'Club/Ciliated',
       'Ciliated', 'NEC', 'Aerocyte capillary EC', 'Transitional capillary EC',
       'General capillary EC', 'Proliferating capillary EC',
       'Vwa1+/Col15a1+ ectopic capillary EC', 'Arterial EC', 'Venous EC',
       'Lymphatic EC', 'Adventitial Fibroblasts', 'Adh7+ Fibroblasts',
       'Lgr5+/Lgr6+ Fibroblasts', 'Lgr5+/Lgr6+ Fibroblasts activated',
       'Lipofibroblasts', 'Lipo/Myofibroblast Transition',
       'Cthrc1+ Myofibroblasts', 'Spp1+ Myofibroblasts', 'Pericytes',
       'Pericytes activated', 'aSMCs', 'vSMCs', 'Mesothelial',
       'Non-classical Monocytes', 'Classical Monocytes',
       'Intermediate Monocytes', 'Transitional Monocytes', 'M2 Macrophages',
       'AM', 'Proliferating AM', 'Prg4+ IM', 'Lyve1+/Cd163+ IM',
       'Lyve1-/Cd163- IM', 'Proliferating IM', 'cDC1', 'cDC2', 'pDC',
       'Proliferating cDC1', 'Proliferating cDC2', 'Megakaryocytes',
       'Basophils', 'Eosinophils', '

### Overall

In [18]:
## list of time points to loop over
time_points = freq_tab['time_point'].unique()

## list of outcomes to loop over
outcome_3 = 'max_body_weight_loss'
outcomes = [outcome_1, outcome_3]

In [19]:
## initialize a DataFrame to store correlation coefficients for both outcomes
correlation_df = pd.DataFrame(index=cell_types)

In [22]:
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

In [23]:
for time_point in time_points:
    
    ## filter the 'freq_tab' for the time point of interest
    df_time_point = freq_tab[freq_tab['time_point'] == time_point]
    
    ## ## initialize a DataFrame to temporarily store results within this for loop
    results = pd.DataFrame(index=cell_types)
    
    for outcome in outcomes:
        ## calculate correlation coefficients for each cell type with the current outcome
        results[outcome + '_spearman_rho'] = [spearmanr(df_time_point[cell_type], df_time_point[outcome])[0] for cell_type in cell_types]
    
    ### add the results to the combined DataFrame with a suffix indicating the time point
    #correlation_df = pd.concat([correlation_df, results.add_suffix(f'_{time_point}')], axis=1)
    
    ## calculate and correct p-values for each outcome
    for outcome in outcomes:
        p_values = [pearsonr(df_time_point[cell_type], df_time_point[outcome])[1] for cell_type in cell_types]
        _, adjusted_p_values, _, _ = multipletests(p_values, method='fdr_bh')
        results[outcome + '_p_value'] = p_values
        results[outcome + '_adjusted_p_value'] = adjusted_p_values
    
    ## add the results to the combined DataFrame with a suffix indicating the time point
    correlation_df = pd.concat([correlation_df, results.add_suffix(f'_{time_point}')], axis=1)

In [24]:
correlation_df.head(5)

,ashcroft_score_spearman_rho_d10,max_body_weight_loss_spearman_rho_d10,ashcroft_score_p_value_d10,ashcroft_score_adjusted_p_value_d10,max_body_weight_loss_p_value_d10,max_body_weight_loss_adjusted_p_value_d10,ashcroft_score_spearman_rho_d21,max_body_weight_loss_spearman_rho_d21,ashcroft_score_p_value_d21,ashcroft_score_adjusted_p_value_d21,max_body_weight_loss_p_value_d21,max_body_weight_loss_adjusted_p_value_d21
AT1,0.477612,0.323208,6.875056e-04,2.315808e-03,1.284107e-02,0.025682,0.329538,0.201191,2.385782e-02,4.362574e-02,0.884490,0.943456
AT2,-0.619906,-0.552230,6.110186e-09,3.910519e-07,1.053483e-05,0.000048,-0.562457,-0.651055,2.228659e-09,2.496258e-08,0.000222,0.001185
AT2 activated,0.188249,0.183826,4.584657e-01,5.536190e-01,8.126755e-01,0.866854,0.092197,-0.039941,2.384291e-01,2.934512e-01,0.499453,0.603113
Krt8+ ADI,0.499049,0.657325,9.216357e-07,9.830781e-06,2.209315e-07,0.000002,0.517119,0.427556,3.854681e-06,2.055830e-05,0.076851,0.136625
Basal,0.149920,0.227104,8.182760e-01,8.182760e-01,3.546257e-01,0.436462,0.359666,0.295983,5.056185e-03,1.198503e-02,0.168677,0.245348


In [25]:
## save results to results dir
correlation_df.to_csv(data_dir + '240830_cell_type_frequency_disease_severity_correlation.csv', sep=',')

In [26]:
## save results to results dir
freq_tab.to_csv(data_dir + '240830_cell_type_frequencies.csv', sep=',')